# Previsão de Falhas Estruturais Aplicadas no <span style="color:red;">Motor Brushless</span>

---

### Integrantes: 
- <span style="color:yellow;">Enrico Magalhães - 220950009</span>;
-  Hugo Samuel - 2023007338;
- <span style="color:brown;">Vinícius Fernandes - 210950064</span>;
- <span style="color:green;">Vitor Belo - 220950023</span>.

### Parâmetros do grupo moto-propulsor utilizado: 
(Como contexto e limitação orçamentária desse estudo, destaca-se que nenhum componente foi adquirido para o projeto. Todos os materiais, e inclusive os ensaios, estavam disponíveis e foram realizados na oficina da equipe Trem Ki Voa Micro. O uso gratuito e facilitado desses componetes foi devido à todos os integrantes do grupo serem membros atuais ou ex-integrantes da equipe.)

- *Motor:* Scorpion SII-4035-450;
- *Hélice:* Master Airscrew 16×10"× 3;
- *Baterias:* LiFePO4 e LIPO;
- *ESC:* ESCTurnigy® Plush-32 120A HV?????? SERÁ?;
- *Receptor:* FrSky® V8FR-II;
-*Regulador de tensão:* Castle Creations® CCBEC10A.

### Sensores:
- *Allegro® ACS712ELCTR*;
- *Célula de carga*;
- *Arduino UNO*;

---

### Repositório no Github: https://github.com/vitorvespoli/MachineLearning_Trabalho1

---

### Definição do problema

A SAE BRASIL é uma associação dedicada ao avanço da engenharia de mobilidade no país. Uma de suas principais iniciativas é o Projeto SAE BRASIL AeroDesign, um programa educacional que desafia estudantes universitários a conceber, projetar, construir e voar aeronaves rádio-controladas, simulando os mesmos desafios enfrentados pela indústria aeronáutica.

Na 26ª Competição SAE Brasil AeroDesign, uma falha no motor impediu o voo da equipe **Trem Ki Voa Micro** (equipe de competição da UFSJ) em uma de suas tentativas oficiais. O evento foi decisivo na pontuação final, resultando na obtenção do vice-campeonato nacional por uma margem de dois pontos em relação ao primeiro colocado.

A partir disso, surgiu a necessidade de desenvolver um modelo capaz de prever falhas com base nos dados de funcionamento do sistema, evitando que falhas passem despercebidas antes de um voo oficial. Dentre as inúmeras falhas que podem interferir no sistema, para a realização desse trabalho, foram escolhidas 3 falhas:

- Desbalanceamento de hélice;
- Hélice quebrada;
- Rotação invertida do motor.

# Obtenção e manipulação dos dados
## Coleta de dados

A coleta de dados do grupo moto-propulsor foi feita através de ensaios em uma bancada de testes estáticos (Figura X). Essa estrutura foi desenvolvida pelo Marcelo Henrique, ex-integrante da equipe Trem Ki Voa Micro e aluno da UFSJ. Para a medição, a bancada utiliza placas Arduino conectadas a sensores que captam as variáveis do sistema: tração, torque, corrente, tensão e potência (Figura X). 

Para monitorar e salvar esses ensaios, utilizamos um aplicativo customizado (Figura X), também criado pelo Marcelo. Esse software nos permitiu acompanhar os parâmetros ao vivo enquanto simulávamos cada condição de voo (hélice desbalanceada, hélice quebrada e rotação invertida) e permitiu a exportação dos dados na forma de arquivos .xlsx. Assim, montamos a base de dados com o histórico de comportamento físico do grupo moto-propulsor. Esse conjunto de informações foi o início para aplicarmos os métodos estatísticos.

## Tratamento de dados
A utilização de uma base de dados própria visa analisar as condições operacionais e os modos de falha do projeto. Contudo, durante as simulações, as leituras dos sensores apresentaram ruídos e pequenas interferências, fatores relacionados ao uso de ligações em protoboards, à vibração estrutural gerada pela força de tração do motor e à forte turbulência do escoamento ao redor da bancada, o que torna a filtragem estatística uma etapa obrigatória para garantir a confiabilidade dos dados. 

Visando atenuar essas inconsistências nos dados, a fim de viabilizar as operações, utilizaram-se as seguintes funções:


###Filtro para correção de dados

Para garantir a confiabilidade dos ensaios e mitigar a presença de ruídos espúrios ou falhas de leitura dos sensores, desenvolveu-se uma rotina de Processamento Digital de Sinais focada no tratamento dos dados reais.

A função desenvolvida realiza, primeiramente, o alinhamento temporal do ensaio. Para isso, o algoritmo adquire a mediana global do sinal de tração e define o instante inicial como o momento em que o valor lido ultrapassa 5% da mediana. Os dados transitórios com ruidos anteriores a este ponto são descartados.

#garantindo a sincronização exata do início do acionamento do motor em todos os gráficos gerados.

Para a correção de falhas sem a distorção do regime transitório natural do motor, optou-se pela exclusão de gabaritos teóricos fixos. Em substituição, o algoritmo utiliza um filtro de tendência central baseado na mediana móvel com uma "janela" estipulada em 51 posições, avaliando o comportamento dos dados ao redor de cada ponto.

A partir dessa curva de tendência, foi estabelecido uma margem de tolerância correspondente a 15% do valor mediano global do sinal. A varredura ocorre de forma pontual: caso a divergência absoluta entre o dado real e a tendência ultrapasse essa margem, o ponto é classificado como um pico (possível falha). Os valores de tolerância e da janela móvel foram definidos a partir da observação do comportamento dinâmico nas bases de dados.

Por fim, os valores extirpados são substituídos por uma interpolação linear, conectando o último ponto válido ao próximo e atuando como um filtro de correção. Para suavizar as transições lineares e evitar angulações agudas, aplicou-se um filtro passa-baixa (Savitzky-Golay) sobre a região interpolada. Além do mais, visando a organização do projeto, ao receber a requisição de tratamento de um arquivo, o algoritmo gera uma pasta com a nomenclatura correspondente ao ensaio e exporta os dados limpos através da plotagem de todas as variáveis (Tração, Torque, Tensão, Corrente e Potência)

In [ ]:
#Pro funcionamento desse código, os dados do ensaio (planilhas) devem estar na mesma pasta desse mesmo código!
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
import os

def processar_filtro_sinais(caminho_arquivo):
    try:
        # Cria pasta
        pasta = caminho_arquivo.rsplit('.', 1)[0]
        os.makedirs(pasta, exist_ok=True)

        # Lê as colunas
        cols = [0, 1, 4, 5, 7]
        if caminho_arquivo.endswith('.csv'):
            try: df = pd.read_csv(caminho_arquivo, usecols=cols, sep=None, engine='python')
            except: df = pd.read_csv(caminho_arquivo, usecols=cols)
        else:
            df = pd.read_excel(caminho_arquivo, usecols=cols)
            
        df.columns = ['Tempo', 'Tracao', 'Tensao', 'Corrente', 'Potencia']
        df = df.dropna()
        
        # Deixa positivo os vlores
        for col in df.columns: 
            df[col] = df[col].abs()
            
        df.reset_index(drop=True, inplace=True) 

        # Sincroniza o tempo
        limite = df['Tracao'].median() * 0.05
        idx = df[df['Tracao'] > limite].index
        if len(idx) == 0: return
        
        df = df.loc[idx[0]:].reset_index(drop=True)
        df['Tempo'] = df['Tempo'] - df['Tempo'].iloc[0] 

        # Define as variáveis
        variaveis = [
            ('Tracao', 'Tração (gf)'),
            ('Tensao', 'Tensão (V)'),
            ('Corrente', 'Corrente (A)'),
            ('Potencia', 'Potência (W)')
        ]

        total_filtrados = 0 

        # Processa cada
        for var, eixo_y in variaveis:
            sinal = df[var]

            # Tendência móvel
            tend_movel = sinal.rolling(window=51, center=True).median().bfill().ffill()

            # Detecta anomalias
            falhas = np.abs(sinal - tend_movel) > (sinal.median() * 0.15)
            total_filtrados += falhas.sum()

            # Interpolação linear
            corrigido = sinal.copy()
            corrigido.loc[falhas] = np.nan
            corrigido = corrigido.interpolate(method='linear').bfill().ffill()

            # Suaviza
            sinal_final = savgol_filter(corrigido, window_length=11, polyorder=3)

            # Plota gráfico
            plt.figure(figsize=(14, 6))
            plt.plot(df['Tempo'], sinal, label='Real', color='orange', alpha=0.6, linewidth=2)

            # Máscara visual
            f_exp = pd.Series(falhas) | pd.Series(falhas).shift(1).fillna(False) | pd.Series(falhas).shift(-1).fillna(False)
            remendos = pd.Series(sinal_final)
            remendos[~f_exp] = np.nan 

            plt.plot(df['Tempo'], remendos, label='Filtro', color='blue', linewidth=3)
            
            plt.title(f'{var}: {caminho_arquivo}', fontsize=14, fontweight='bold')
            plt.xlabel('Tempo (s)', fontsize=12)
            plt.ylabel(eixo_y, fontsize=12)
            plt.legend()
            plt.grid(True, linestyle='--', alpha=0.7)
            plt.tight_layout()
            
            # Salva imagem
            plt.savefig(os.path.join(pasta, f"{var}.png"), dpi=300)
            plt.close()
            
        # Exibe contagem
        print(f"\nQuantidade de dados filtrados: {total_filtrados}")
            
    except Exception:
        pass 

# Solicita arquivo
nome_digitado = input("Digite o nome da planilha: ")
if not (nome_digitado.endswith('.xlsx') or nome_digitado.endswith('.csv')):
    nome_digitado += '.xlsx'
processar_filtro_sinais(nome_digitado)